In [14]:
import truststore

truststore.inject_into_ssl()

In [15]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

In [17]:
class SubState(TypedDict):
    input_text: str
    translated_text: str

In [18]:
subgraph_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

In [19]:
def transalate_text(state: SubState):
    prompt = f"""
Transalate the following text into hindi. {state['input_text']}
"""
    transalted_text = subgraph_llm.invoke(prompt).content
    return {"translated_text": transalted_text}

In [20]:
graph = StateGraph(SubState)

graph.add_node("transalate_text", transalate_text)

graph.add_edge(START, "transalate_text")
graph.add_edge("transalate_text", END)

subgraph = graph.compile()

In [21]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    answer_hin: str

In [22]:
parent_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

In [23]:
def generate_answer(state: ParentState):
    answer = parent_llm.invoke(f"Answer this question: {state['question']}").content
    return {"answer_eng": answer}

In [24]:
def translate_answer(state: ParentState):
    result = subgraph.invoke({"input_text": state["answer_eng"]})
    return {"answer_hin": result["translated_text"]}

In [25]:
graph = StateGraph(ParentState)

graph.add_node("generate_answer", generate_answer)
graph.add_node("translate_answer", translate_answer)

graph.add_edge(START, "generate_answer")
graph.add_edge("generate_answer", "translate_answer")
graph.add_edge("translate_answer", END)

parentgraph = graph.compile()

In [26]:
parentgraph.invoke({"question": "What is lol"})

{'question': 'What is lol',
 'answer_eng': '"LOL" is an abbreviation that stands for "Laugh Out Loud." It is a common expression used in digital communication, such as text messages, social media, and online forums, to indicate that something is funny or amusing. When someone types "LOL," they are essentially saying that they found something humorous or entertaining. The phrase has become a widely recognized and frequently used term in internet slang.',
 'answer_hin': '"एलओएल" एक संक्षिप्त नाम है जो "लाफ आउट लाउड" के लिए खड़ा है। यह एक सामान्य अभिव्यक्ति है जो डिजिटल संचार में उपयोग की जाती है, जैसे कि टेक्स्ट संदेश, सोशल मीडिया, और ऑनलाइन मंचों में, यह दर्शाने के लिए कि कुछ मजाकिया या मनोरंजक है। जब कोई "एलओएल" टाइप करता है, तो वे मूल रूप से यह कह रहे होते हैं कि उन्हें कुछ हास्यमय या मनोरंजक लगा। यह वाक्यांश इंटरनेट स्लैंग में एक व्यापक रूप से मान्यता प्राप्त और बार-बार उपयोग किया जाने वाला शब्द बन गया है।'}

In [27]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    answer_hin: str

In [30]:
def transalate_text_1(state: ParentState):
    prompt = f"""
Transalate the following text into hindi. {state['answer_eng']}
"""
    transalted_text = subgraph_llm.invoke(prompt).content
    return {"answer_hin": transalted_text}

In [31]:
graph = StateGraph(ParentState)

graph.add_node("transalate_text", transalate_text_1)

graph.add_edge(START, "transalate_text")
graph.add_edge("transalate_text", END)

subgraph = graph.compile()

In [32]:
def generate_answer(state: ParentState):
    answer = parent_llm.invoke(f"Answer this question: {state['question']}").content
    return {"answer_eng": answer}

In [33]:
graph = StateGraph(ParentState)

graph.add_node("generate_answer", generate_answer)
graph.add_node("subgraph", subgraph)

graph.add_edge(START, "generate_answer")
graph.add_edge("generate_answer", "subgraph")
graph.add_edge("subgraph", END)

parentgraph = graph.compile()

In [34]:
parentgraph.invoke({"question": "What is lol"})

{'question': 'What is lol',
 'answer_eng': '"LOL" is an abbreviation that stands for "Laugh Out Loud." It is a common expression used in digital communication, such as text messages, social media, and online forums, to indicate that something is funny or amusing. When someone types "LOL," they are essentially saying that they found something humorous or entertaining. The phrase has become a widely recognized and frequently used term in internet slang.',
 'answer_hin': '"एलओएल" एक संक्षिप्त नाम है जिसका अर्थ है "जोर से हंसना"। यह एक सामान्य अभिव्यक्ति है जो डिजिटल संचार में उपयोग की जाती है, जैसे कि टेक्स्ट संदेश, सोशल मीडिया, और ऑनलाइन मंचों में, यह दर्शाने के लिए कि कुछ मजाकिया या मनोरंजक है। जब कोई "एलओएल" टाइप करता है, तो वे मूल रूप से यह कह रहे होते हैं कि उन्हें कुछ हास्यमय या मनोरंजक लगा। यह वाक्यांश इंटरनेट स्लैंग में एक व्यापक रूप से मान्यता प्राप्त और बार-बार उपयोग किया जाने वाला शब्द बन गया है।'}